## MiddleWare


In [26]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

### Summarization MiddleWare

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


agent=create_agent(
    model='groq:qwen/qwen3-32b',
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:qwen/qwen3-32b',
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)



In [ ]:
### Run With A thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
questions=[
    "What is 2+2",
    "What is the capital of India",
    "What is 2* 3",
    "What is 15-7",
]

for q in questions:
    response=agent.invoke({
        "messages":[
            HumanMessage(content=q)
        ]
    },config)
    print(response)
    print(len(response['messages']))

{'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='e7c876bf-7123-4b19-826a-aca9ba84241f'), AIMessage(content='<think>\nOkay, so the question is "What is 2+2?" Hmm, that seems straightforward, but maybe I should think through it carefully. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations in mathematics, right? When you add two numbers together, you\'re combining their quantities. So, 2 plus 2 would be combining two units with another two units. Let me visualize this: if I have two apples and someone gives me two more apples, how many apples do I have in total? That would be four apples. So, 2+2 equals 4.\n\nWait, but maybe I should consider different number systems? Like, in base 10, which is the standard system we use daily, 2+2 is indeed 4. But what if someone is thinking in a different base? For example, in base 3, the digits go 0, 1, 2, and then 10. So, adding 2+2 in base 3 would be 11, which is 

In [17]:
### Token Size Based trigger
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 Star, $350/night,spa, pool,gym
    2. City Palace - 4 Star, $200/night, pool, gym
    3. Hotel Lake Star - 3.5 Star, $200/night, pool"""

agent=create_agent(
    model='google_genai:gemini-2.5-flash',
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='google_genai:gemini-2.5-flash',
            trigger=("tokens",550),
            keep=("tokens",200),
        )        
                ]
)

### Run With A thread id
config={"configurable":{"thread_id":"test-1"}}


def token_counter(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [ ]:
#Run test

cities=['Paris','Dubai','London','America','Singapore']

for city in cities:
    response=agent.invoke(
        {
            "messages":[
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )
    tokens=token_counter(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

## Human in the loop

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver




def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id """
    return f"Email Content for ID: {email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject}"

agent=create_agent(
    model='google_genai:gemini-flash-latest',
    tools=[
        read_email_tool,send_email_tool,
    ],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{"allowed_decisions":["approve","edit","reject"]},
                "read_email_tool":False 
            }
        )
    ]
)


config={"configurable":{"thread_id":"test-1"}}

result=agent.invoke({"messages":[HumanMessage(content="Send an email to john@test.com with subject 'Hello' and body 'How are you?' ")]},config)


result

{'messages': [HumanMessage(content="Send an email to john@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='b4431aa0-87ae-4d0b-8bad-f540ea3529d4'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "body": "How are you?", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'fvclgrlk': 'EosDCogDAQw51sclzTlHcx79+Yo5WufeTK3YxZV0bcneORLvO6WAaZymh5gN+pVsLaHW5cCA2ZmjYiWfH1STQV3LghjS4dbLAcGxEEkx/rhRWYfvRvtdAXzrJ59YITo0QdQvqn26UU6RPYuwEU6aa8u8MlfiLPClMGZbEr1AUSr/SsqPJlxjHc05zYayz4nKbqaxST2qGthPIf3P+lvHd0urhik+3ECfwz+aFiw6CWkRDhzfKj0+HMyxQuOd2gMFE2e9PtE0Rv5yHJL7StwsQYl0BjIjcDKjqk2cXo58rZl8dOlmuALME6XSIkOELuu5G7shZtyV1j0GxtQ7Wkdlxf0zSlkl2L63DYZ5E3bIE7WcvmWykOKNOrCrHgbjy+1dPMH1L0DTy6DdVzYZ8r3drRIKedb/8VBHEFd3/tU2Rty2HFSlbpPn7YDzqU3fwq//vkdJdhajoP+jrILgYzqysCz5cmFHtNN56+s5KGZgaYNqgx6Tlrf0vY4FW94J/kmixBr7SNCEXW/QsunlMEA='}}, response_metadata={

In [ ]:
# Step 2 Approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused!! Approving...")
    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"Result: {result['messages'][-1].content}")

Paused!! Approving...


NameError: name 'Command' is not defined